# Seurat (R) tutorial
### How to run Ouroboros on a Seurat (R) object

Ouroboros is a python package, so running it on a Seurat object requires saving your raw counts as a CSV file and running the command-line implementation of Ouroboros on the saved CSV. The ouroboros output can then be loaded into back into R.


Dependencies:
- Seurat
- SeuratObject
- Matrix
- Zellconverter 
- SingleCellExperiment


In [ ]:
suppressPackageStartupMessages({
  library(Seurat)
  library(SeuratObject)
  library(Matrix)
  library(zellkonverter)
  library(SingleCellExperiment)
  library(SeuratData)
})

We'll demonstrate Ouroboros using **pbmc3k**, a standard dataset of 2,700 PBMCs
(~13,700 genes) that ships as raw counts. We subset it to 100 cells to remove computational time and written object size. 

In [10]:
InstallData("pbmc3k")
data("pbmc3k")

pbmc3k   # ~2700 cells x ~13714 genes, raw counts in the RNA assay

Installing package into ‘/projects/pangen/analysis/hmac/applications/miniconda3/lib/R/library’
(as ‘lib’ is unspecified)



In [12]:
set.seed(330)
cells <- colnames(GetAssayData(pbmc3k, assay = "RNA", layer = "counts"))
pbmc100 <- pbmc3k[, sample(cells, 100)]

Next write the raw counts as a Scanpy h5ad object to run ouroboros on. 
    p.s. this tends to take a long time (~ 15 minutes)

In [14]:
# Seurat -> SingleCellExperiment
sce <- as.SingleCellExperiment(pbmc100)

# confirm a raw-counts assay is present
assayNames(sce)          # expect "counts" (and usually "logcounts")

# write .h5ad with RAW counts in .X
writeH5AD(sce, "R_example_output/pbmc_small.h5ad", X_name = "counts")

[1] "counts"    "logcounts"

There is still some formatting of that h5ad that needs to be done so Ouroboros will recognize it. Run this in a terminal (replace the h5ad path with your own):

```bash
conda activate ouroboros_env

# go to your output dir
cd R_example_output
```

```bash
python - <<'PY'
import h5py, numpy as np, anndata as ad
path = "pbmc_small.h5ad"
vlen = h5py.special_dtype(vlen=str)
with h5py.File(path, "r+") as f:
    if "layers/raw_counts" not in f:
        f.copy("X", "layers/raw_counts")

    # strip modern 'dict' tags (already done, but harmless to repeat)
    def fix_dict(name, obj):
        if isinstance(obj, h5py.Group):
            et = obj.attrs.get("encoding-type")
            et = et.decode() if isinstance(et, (bytes, bytearray)) else et
            if et == "dict":
                del obj.attrs["encoding-type"]; obj.attrs.pop("encoding-version", None)
    f.visititems(fix_dict)

    # flatten obs/var to index-only: drop categorical/group columns the old reader can't parse
    for key in ("obs", "var"):
        if key not in f: continue
        g = f[key]
        idx = g.attrs.get("_index", "_index")
        idx = idx.decode() if isinstance(idx, (bytes, bytearray)) else idx
        for col in list(g.keys()):
            if col != idx:
                del g[col]
        g.attrs.create("column-order", np.array([], dtype=vlen))
PY
```


Double check it worked with this: 
```bash
python - <<'PY'
import anndata as ad
a = ad.read_h5ad("pbmc_small.h5ad")
print("OK:", a.shape, "| genes:", list(a.var_names[:3]), "| raw_counts:", "raw_counts" in a.layers)
PY
```

Then run Ouroboros on test object on command line

Call:
```bash

ouroboros \
    --data pbmc_small.h5ad \
    --data_type h5ad \
    --outdir .
```